###1.从公开数据集提取所需信息

In [2]:
import json
import pandas as pd
from collections import defaultdict

# 文件路径
file_path = 'D:/研究生论文/小论文2/代码/yelp_academic_dataset_review.json'

# 第一步：统计每个 business_id 出现的次数
business_review_count = defaultdict(int)

# 先做第一轮遍历，统计每个 business_id 的评论数量
with open(file_path, 'r', encoding='utf-8') as f:
    for line in f:
        review = json.loads(line)
        business_review_count[review['business_id']] += 1

# 找出评论最多的 business_id
most_reviewed_business = max(business_review_count, key=business_review_count.get)
print(f"评论最多的 business_id 是：{most_reviewed_business}，评论数为：{business_review_count[most_reviewed_business]}")

# 第二步：提取该 business_id 的所有评论
matched_reviews = []

with open(file_path, 'r', encoding='utf-8') as f:
    for line in f:
        review = json.loads(line)
        if review['business_id'] == most_reviewed_business:
            matched_reviews.append(review)

# 转换为 DataFrame 并选择字段
df = pd.DataFrame(matched_reviews)
columns_to_save = ['review_id', 'user_id', 'stars', 'useful', 'funny', 'cool', 'text', 'date']
df = df[columns_to_save]

# 保存为 Excel 文件
output_file = 'D:/研究生论文/小论文2/代码/filtered_yelp_most_reviewed.xlsx'
df.to_excel(output_file, index=False)

print(f"保存成功，共提取 {len(df)} 条评论，已保存为 {output_file}")


评论最多的 business_id 是：RESDUcs7fIiihp38-d6_6g，评论数为：10417
保存成功，共提取 10417 条评论，已保存为 D:/研究生论文/小论文2/代码/filtered_yelp_most_reviewed.xlsx


#数据预处理

In [4]:
import pandas as pd
import re

# 读取Excel文件
#file_path = 'D:/研究生论文/小论文2/代码/01数据预处理/02输出结果-预处理/桃花马上请长缨/6.筛选后文档.xlsx'
file_path = 'D:/研究生论文/小论文2/代码/01数据预处理/00data-初始-未合并集数/filtered_yelp_most_reviewed.xlsx'
data = pd.read_excel(file_path)  # , engine='openpyxl'或者 engine='xlrd'
#其中 openpyxl 用于.xlsx文件格式，xlrd 用于.xls文件格式。

# 定义函数去除 [] 和 () 中的内容
def remove_brackets(text):
    text = re.sub(r'\[.*?\]', '', str(text))  # 去除 []
    text = re.sub(r'\(.*?\)', '', text)  # 去除 ()
    text = re.sub(r'\【.*?\】', '', text)  # 去除 【】
    return text

# 在"内容这一列"上应用函数
data['内容'] = data['内容'].apply(remove_brackets)

# 筛选字符小于15个字符的行
filtered_data = data[data['内容'].apply(lambda x: len(str(x)) >= 15)]

#data = data[data['内容'].notna() & (data['内容'] != '')]

# 保存结果到新的Excel文件
output_path = 'D:/研究生论文/小论文2/代码/01数据预处理/02输出结果-预处理/Yelp/5.数据预处理.xlsx'
filtered_data.to_excel(output_path, index=False)


#有用性

In [2]:
import pandas as pd
file_path = "D:/研究生论文/小论文2/代码/01数据预处理/02输出结果-预处理/Yelp/5.数据预处理.xlsx"
df = pd.read_excel(file_path)

# 假设df是一个pandas DataFrame，其中包含了用户等级、是否VIP、点赞数和回复数
# 我们假设Excel文件中的列名是中文，这里进行相应的替换
# 添加是否VIP的布尔值权重，假设VIP为'是'，非VIP为'否'
#df['vip_weight'] = df['VIP'].apply(lambda x: 1 if x == '是' else 0)

# 标准化用户等级、点赞数和回复数
#df['user_level_norm'] = (df['等级'] - df['等级'].min()) / (df['等级'].max() - df['等级'].min())
df['likes_norm'] = (df['点赞数'] - df['点赞数'].min()) / (df['点赞数'].max() - df['点赞数'].min())
df['replies_norm'] = (df['回复数'] - df['回复数'].min()) / (df['回复数'].max() - df['回复数'].min())

# 计算权威性得分（这里假设VIP的权重和用户等级同等重要）
#df['authority_score'] = (df['user_level_norm'] + df['vip_weight'] * 2) / 3
#df['authority_score'] = (df['user_level_norm'] + df['vip_weight'] ) / 2

# 计算认同度得分（这里假设点赞数和回复数同等重要）
df['approval_score'] = (df['likes_norm'] + df['replies_norm']* 2) / 3


# 定义权重
#authority_weight = 0
#approval_weight = 1


# 计算综合指标
#df['useful_index'] = (df['authority_score'] * authority_weight) + (df['approval_score'] * approval_weight)
df['useful_index'] = df['approval_score']

# 输出新的DataFrame
df.to_excel("D:/研究生论文/小论文2/代码/01数据预处理/02输出结果-预处理/Yelp/6.赋权后文档.xlsx", index=False)  # 用户需要将此路径替换为希望保存的新文件的实际路径


In [3]:

stats = df['useful_index'].describe()
print(stats)


count    10416.000000
mean         0.005374
std          0.022806
min          0.000000
25%          0.000000
50%          0.000000
75%          0.004193
max          0.990530
Name: useful_index, dtype: float64


In [4]:
import pandas as pd

# 根据 useful_index 列的数值进行筛选
filtered_df = df[df['useful_index'] >= 0.004193]   #50%处的值

# 输出筛选后的结果
print(filtered_df)

# 如果需要保存到新的 Excel 文件
filtered_df.to_excel("D:/研究生论文/小论文2/代码/01数据预处理/02输出结果-预处理/Yelp/7.最终有效评论.xlsx", index=False)


                    review_id                 user_id  stars  点赞数  回复数  cool  \
1      JlNnsvMPLK_1-X2hwzK24w  IS9yw8P2uAPBX6FNLLX4KA      4   39   21    29   
2      hBkoWffORRb6aqKhC_Li2A  uZdFsE_aHbFBChgN6Xa8tw      4    1    1     1   
13     nb6KNON8Rulne5Dkm7tIMQ  5UtrlKrpROenKBYvJeM-BA      5    3    1     1   
18     3pZ02SGyItAGYRWGIJedBg  5Hym66RYRlDkFkvYpw04OQ      5    1    1     1   
19     OCLDKgX8vjhRL-uDcDkP2g  DKolrsBSwMTpTJL22dqJRQ      3    2    1     0   
...                       ...                     ...    ...  ...  ...   ...   
10406  1M8cypfOEn9FwpdXN3qaVw  CYBHRsbg9eGP9S1pnqGOhQ      4    1    1     0   
10407  _xEeTl3o9nbl949_NKFsdQ  deL6e_z9xqZTIODKqnvRXQ      5    4    1     3   
10412  gbBau-2wy3_kNr2y6dEa1Q  c-j3TV1F8rI6bQUD6nqGPQ      4    3    0     3   
10414  75nzyA96_BgVrpflweAA3w  6rEG-G4syq5IvWti4tyPXA      4    1    1     2   
10415  mMa_YQNBJfuh_Nw_x81jlw  GsALS1y9wJoBRJTEzJiISg      4    1    1     1   

                                       